In [ ]:
!pip install stable-baselines3
!pip install aquacrop
!git clone https://github.com/aquacropos/aquacrop-gym.git
%cd aquacrop-gym
!pip install tensorflow



Cloning into 'aquacrop-gym'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 129 (delta 26), reused 117 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 18.46 MiB | 20.80 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/aquacrop-gym/aquacrop-gym/aquacrop-gym/aquacrop-gym


In [ ]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium import Env
from gymnasium.spaces import Discrete, Box
import numpy as np
import random
import aquacropgym

In [ ]:
import aquacrop

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather, get_filepath
weather_file_path = get_filepath('tunis_climate.txt')


In [ ]:
from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent, IrrigationManagement
from aquacrop.utils import prepare_weather, get_filepath
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium.spaces import Box

class AquaCropEnv(gym.Env):
    def __init__(self):
      self.action_space = Box(
          low=np.array([0.0]),
          high=np.array([999.0]),   # max mm/day
          dtype=np.float32
      )

      self.observation_space = Box(
          low=np.array([0, 0, 0], dtype=np.float32),
          high=np.array([1, 100, 100000], dtype=np.float32),
          dtype=np.float32
      )

      self.weather_df = prepare_weather(get_filepath("tunis_climate.txt"))
      self.start_date = pd.Timestamp("1979-10-01")
      self.end_date = pd.Timestamp("1980-05-30")
      self.day = 0
      self.irrigation_schedule = []

    def _make_model(self):
        schedule = pd.DataFrame(
            self.irrigation_schedule,
            columns=["Date", "Depth"]
        )

        irrigation = IrrigationManagement(
            irrigation_method=3,   # predefined schedule
            Schedule=schedule
        )

        model = AquaCropModel(
            sim_start_time=self.start_date.strftime("%Y/%m/%d"),
            sim_end_time=self.end_date.strftime("%Y/%m/%d"),
            weather_df=self.weather_df,
            soil=Soil(soil_type="SandyLoam"),
            crop=Crop("Wheat", planting_date="10/01"),
            initial_water_content=InitialWaterContent(value=["FC"]),
            irrigation_management=irrigation
        )

        return model

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.day = 0
        self.irrigation_schedule = []

        self.model = self._make_model()
        self.model.run_model(num_steps=1, initialize_model=True)

        return self._get_obs(), {}

    def step(self, action):
      irrigation = float(action[0])
      irrigation = np.clip(irrigation, 0, 30)

      self.schedule.loc[self.day, "Depth"] = irrigation
      self.total_water += irrigation

      self.model.run_model(num_steps=1)

      storage = self.model.get_water_storage()
      flux = self.model.get_water_flux()

      obs = np.array([
          storage["th1"].iloc[-1],
          self.model._init_cond.canopy_cover,
          self.model._init_cond.biomass
      ], dtype=np.float32)

      reward = -0.01 * irrigation

      terminated = self.day >= 179 or self.model._init_cond.crop_mature
      truncated = False

      if terminated:
          results = self.model.get_simulation_results()
          final_yield = results["Dry yield (tonne/ha)"].iloc[-1]
          reward += final_yield * 10

      self.day += 1

      return obs, reward, terminated, truncated, {}

    def _get_obs(self):
      init = self.model._init_cond

      theta_first_layer = init.th[0]

      canopy_cover = getattr(init, "canopy_cover", 0)
      biomass = getattr(init, "biomass", 0)

      return np.array([
          theta_first_layer,
          canopy_cover,
          biomass
      ], dtype=np.float32)

In [ ]:
model = AquaCropModel(
            sim_start_time=f"{1979}/10/01",
            sim_end_time=f"{1985}/05/30",
            weather_df=prepare_weather(weather_file_path),
            soil=Soil(soil_type='SandyLoam'),
            crop=Crop('Wheat', planting_date='10/01'),
            initial_water_content=InitialWaterContent(value=['FC']),
        )
# 1 день симуляции
model.run_model(num_steps=1)

# влажность по soil compartments / слоям
theta_layers = model._init_cond.th

# первый слой
theta_first_layer = theta_layers[0]

print(theta_first_layer)

0.19077646219099564


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
env=AquaCropEnv()
episodes=10
for episode in range( 1, episodes+1) :
  state = env.reset()
  done = False
  score = 0
  while not done:
    action = env.action_space.sample()
    n_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    score += reward
  print('Episode: {} Score: {}' .format(episode, score))

/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(

KeyboardInterrupt: 

In [ ]:
from stable_baselines3 import PPO

policy_kwargs = dict(
    net_arch=[64, 64]  # 2 hidden layers, 64 neurons each
)

agent = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.0003,
    n_steps=512,
    batch_size=64,
    gamma=0.99
)

agent.learn(total_timesteps=100)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 180      |
|    ep_rew_mean     | -0.728   |
| time/              |          |
|    fps             | 20       |
|    iterations      | 1        |
|    time_elapsed    | 25       |
|    total_timesteps | 512      |
---------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
obs, info = env.reset()
done = False
total_reward = 0

while not done:
    action, _ = agent.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated
    total_reward += reward

print("Total reward:", total_reward)

Total reward: 0.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
obs, info = env.reset()

for i in range(10):
    action = np.array([30.0], dtype=np.float32)
    obs, reward, terminated, truncated, info = env.step(action)

    print(i, obs, reward)
    print("irr_cum:", env.model._init_cond.irr_cum)

0 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0
1 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0
2 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


3 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0
4 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0
5 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


6 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0
7 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0
8 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0
9 [0.3726 0.     0.    ] -0.3
irr_cum: 25.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent, IrrigationManagement
from aquacrop.utils import prepare_weather, get_filepath


class AquaCropIrrigationEnv(gym.Env):
    def __init__(self):
        super().__init__()

        self.weather = prepare_weather(get_filepath("tunis_climate.txt"))

        self.start_date = "1979-10-01"
        self.end_date = "1980-05-30"
        self.max_days = 180

        self.action_space = spaces.Box(
            low=np.array([0.0], dtype=np.float32),
            high=np.array([30.0], dtype=np.float32),
            dtype=np.float32
        )

        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 100000.0], dtype=np.float32),
            dtype=np.float32
        )

        self.day = 0
        self.total_water = 0.0
        self.irrigation_schedule = None
        self.model = None

    def _make_schedule(self):
        return pd.DataFrame({
            "Date": pd.date_range(self.start_date, periods=240, freq="D"),
            "Depth": np.zeros(240, dtype=float)
        })

    def _build_model(self):
        schedule_df = self.irrigation_schedule.copy()

        irrigation = IrrigationManagement(
            irrigation_method=3,
            Schedule=schedule_df
        )

        model = AquaCropModel(
            sim_start_time="1979/10/01",
            sim_end_time="1980/05/30",
            weather_df=self.weather,
            soil=Soil("SandyLoam"),
            crop=Crop("Wheat", planting_date="10/01"),
            initial_water_content=InitialWaterContent(value=["FC"]),
            irrigation_management=irrigation
        )

        return model

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.day = 0
        self.total_water = 0.0
        self.irrigation_schedule = self._make_schedule()

        self.model = self._build_model()
        self.model.run_model(num_steps=1)

        return self._get_obs(), {}

    def step(self, action):
        irrigation = float(np.asarray(action).reshape(-1)[0])
        irrigation = float(np.clip(irrigation, 0.0, 30.0))

        self.irrigation_schedule.loc[self.day, "Depth"] = irrigation
        self.total_water += irrigation

        self.model = self._build_model()
        self.model.run_model(num_steps=self.day + 1)

        obs = self._get_obs()

        terminated = self.day >= self.max_days - 1
        truncated = False

        reward = -0.01 * irrigation

        if terminated:
            self.model.run_model(till_termination=True)
            results = self.model.get_simulation_results()
            final_yield = float(results["Dry yield (tonne/ha)"].iloc[-1])

            reward += final_yield * 10
            reward -= self.total_water * 0.02

        self.day += 1

        return obs, float(reward), terminated, truncated, {}

    def _get_obs(self):
        th1 = float(self.model._init_cond.th[0])
        canopy = float(self.model._init_cond.canopy_cover)
        biomass = float(self.model._init_cond.biomass)

        return np.array([th1, canopy, biomass], dtype=np.float32)

    def render(self):
        print(f"Day: {self.day}, Total water: {self.total_water}")

In [ ]:
env = AquaCropIrrigationEnv()
obs, info = env.reset()

for i in range(10):
    action = np.array([30.0], dtype=np.float32)
    obs, reward, terminated, truncated, info = env.step(action)

    print("day:", i)
    print("obs:", obs)
    print("reward:", reward)
    print("irr_cum:", env.model._init_cond.irr_cum)
    print("th1:", env.model._init_cond.th[0])
    print()

day: 0
obs: [0.3726 0.     0.    ]
reward: -0.3
irr_cum: 25.0
th1: 0.3726

day: 1
obs: [0.377 0.    0.   ]
reward: -0.3
irr_cum: 50.0
th1: 0.377

day: 2
obs: [0.3781 0.     0.    ]
reward: -0.3
irr_cum: 75.0
th1: 0.37810000000000005

day: 3
obs: [0.3704 0.     0.    ]
reward: -0.3
irr_cum: 100.0
th1: 0.3704



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 4
obs: [0.3704 0.     0.    ]
reward: -0.3
irr_cum: 125.0
th1: 0.3704

day: 5
obs: [0.377 0.    0.   ]
reward: -0.3
irr_cum: 150.0
th1: 0.377

day: 6
obs: [0.3792 0.     0.    ]
reward: -0.3
irr_cum: 175.0
th1: 0.37920000000000004



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 7
obs: [0.3814 0.     0.    ]
reward: -0.3
irr_cum: 200.0
th1: 0.3814

day: 8
obs: [0.3759 0.     0.    ]
reward: -0.3
irr_cum: 225.0
th1: 0.37589999999999996

day: 9
obs: [0.3737 0.     0.    ]
reward: -0.3
irr_cum: 250.0
th1: 0.3737



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from stable_baselines3 import PPO

env = AquaCropIrrigationEnv()

agent = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    n_steps=128,
    batch_size=64
)

agent.learn(total_timesteps=200)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

----------------------------
| time/              |     |
|    fps             | 3   |
|    iterations      | 1   |
|    time_elapsed    | 34  |
|    total_timesteps | 128 |
----------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


AttributeError: 'numpy.ndarray' object has no attribute 'Date'

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
obs, info = env.reset()
done = False

while not done:
    action, _ = agent.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

print("Total water:", env.total_water)
print("Final reward:", reward)

KeyboardInterrupt: 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
